# SoulX optimization evidence audit

Executed CPU companion to [the validation report](../EVIDENCE_VALIDATION_2026-09-07.md). This notebook rechecks historical JSON measurements; it does not run inference, load weights, reserve GPU memory, or measure new performance.

## TL;DR

The recorded native-portrait TRT ten-job median is about 28.57 useful FPS. Ten simultaneous 25-FPS speakers require 250 useful FPS before headroom. Wire FPS and held/generated-idle frames cannot be substituted for useful speech throughput.

## Context & Methods

Sources: committed benchmark JSON under `benchmarks/implementation`; source snapshot `ac2c2bbb7308f87880c2ae4b01563d1e4d203c02`. Grain: one offline row per `(file, mode, repeat)`; phase subrows are individual generated chunks. Repeated chunks within a run are correlated, not independent trials. The matched alternating Torch A/B and separate Torch/TRT runs have different comparison strength.

Checks: SHA-256 provenance; unique row keys; expected repetitions, useful frame counts and chunk counts; finite/positive timing; row FPS arithmetic; medians reconciled to stored summaries; phase means over the exact chunk population; correct call-counter denominators; final fallback counter checks; capacity and Amdahl arithmetic. All assertions must pass. Percentiles in saved small-repeat files are descriptive. The soak counter snapshot precedes its trailing receiver collection and detailed turn stats retain only the latest 64 turns.

No external packages beyond the notebook runner are required. All inference/backend/quality claims remain historical and conditional on the source/profile recorded in each input.

In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib, json, math, statistics, subprocess, ast
from datetime import datetime, timezone

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "soulx_rtc/engine.py").exists())
DATA = ROOT / "benchmarks/implementation"
offline_names = [
    "portrait-allocator-ab.json",
    "portrait-ten-jobs.json",
    "portrait-trt-ten-jobs.json",
    "portrait-9x16-staged.json",
    "square-trt-combined-reference.json",
]
call_names = [
    "calls-generated-idle-soak30m.json",
    "calls-h264-trt-c10.json",
    "calls-final-staged-c1.json",
]
reports = {n: json.loads((DATA/n).read_text()) for n in offline_names + call_names}
print("Execution UTC:", datetime.now(timezone.utc).isoformat())
print("Input files:", len(reports))
for name in reports:
    print(name, hashlib.sha256((DATA/name).read_bytes()).hexdigest())


Execution UTC: 2026-09-07T04:49:15.550210+00:00
Input files: 8
portrait-allocator-ab.json 2ed4714a163627318cb63adeac068213009afdc1c3d92c16a3b9e20d4acf3ff0
portrait-ten-jobs.json b545da9468091a4eb442e1bea61ef7daa814d5690db5bc8823912fdfc1582d22
portrait-trt-ten-jobs.json 262d9ebb2a6c5af796651438c04ceb4a0b6ed5c1a53d5037d1a9abd8a15dead9
portrait-9x16-staged.json fedcd81014c850f3dc6e3013e77a9e62ca7d6a7beb9ac8de61c9d0c8cf894f3f
square-trt-combined-reference.json bdc28de9ce6df25ae84dc598f433766f98329bdc31353f913f80e10199c3ebec
calls-generated-idle-soak30m.json c233406c4553bf487bcbadf11d58547a5085c5f9e95309a3b7977486121f6390
calls-h264-trt-c10.json 39267b8bd8bcf68d2c13620bfe352982f6bbff7973e056265f034f50e193ec2d
calls-final-staged-c1.json 288cbc7b29a12ee20aa5956b3efcbd4ef8478f53344e2e4008e4c7aed82a491d


## Data QA: repetitions, grain, frame accounting and summary reconciliation

In [2]:
expected_rows = {
    "portrait-allocator-ab.json": 10, "portrait-ten-jobs.json": 5,
    "portrait-trt-ten-jobs.json": 5, "portrait-9x16-staged.json": 3,
    "square-trt-combined-reference.json": 3,
}
medians = {}
phase_means = {}
for name in offline_names:
    d = reports[name]
    rows = d["rows"]
    assert len(rows) == expected_rows[name], (name, len(rows))
    keys = [(r["mode"], r["repeat"]) for r in rows]
    assert len(keys) == len(set(keys)), (name, "duplicate row key")
    grouped = defaultdict(list)
    for r in rows:
        assert all(math.isfinite(r[k]) and r[k] > 0
                   for k in ("wall_s", "aggregate_fps", "useful_frames"))
        assert math.isclose(r["aggregate_fps"],
                            r["useful_frames"]/r["wall_s"], rel_tol=1e-10)
        assert r["useful_frames"] == 250 * d["config"]["sessions"]
        assert len(r["chunks"]) == 11 * d["config"]["sessions"]
        assert all(c["batch"] == 1 for c in r["chunks"])
        grouped[r["mode"]].append(r)
    for mode, group in sorted(grouped.items()):
        value = statistics.median(r["aggregate_fps"] for r in group)
        assert math.isclose(value, d["summary"][mode]["median_fps"], rel_tol=1e-10)
        medians[(name, mode)] = value
        chunks = [c for r in group for c in r["chunks"]]
        stage_keys = set(chunks[0]["stages_ms"])
        assert all(set(c["stages_ms"]) == stage_keys for c in chunks)
        assert all(math.isfinite(v) and v >= 0
                   for c in chunks for v in c["stages_ms"].values())
        phase_means[(name, mode)] = {
            k: statistics.mean(c["stages_ms"][k] for c in chunks)
            for k in sorted(stage_keys)
        }
        print(f"{name} {mode}: rows={len(group)}, chunks={len(chunks)}, "
              f"useful_frames={sum(r['useful_frames'] for r in group)}, "
              f"median_FPS={value:.6f}, "
              f"median_wall_s={statistics.median(r['wall_s'] for r in group):.6f}")
print("PASS: unique keys, counts, finite timings, FPS arithmetic and stored medians")


portrait-allocator-ab.json baseline: rows=5, chunks=55, useful_frames=1250, median_FPS=26.066840, median_wall_s=9.590729
portrait-allocator-ab.json real: rows=5, chunks=55, useful_frames=1250, median_FPS=27.508525, median_wall_s=9.088092
portrait-ten-jobs.json real: rows=5, chunks=550, useful_frames=12500, median_FPS=27.599924, median_wall_s=90.579961
portrait-trt-ten-jobs.json real: rows=5, chunks=550, useful_frames=12500, median_FPS=28.569589, median_wall_s=87.505635
portrait-9x16-staged.json real: rows=3, chunks=33, useful_frames=750, median_FPS=10.883477, median_wall_s=22.970601
square-trt-combined-reference.json real: rows=3, chunks=33, useful_frames=750, median_FPS=44.261183, median_wall_s=5.648290
PASS: unique keys, counts, finite timings, FPS arithmetic and stored medians


## Results: gains, phase costs and upper bounds

The A/B percentage below is a ratio of run medians, not a mean of paired percentage differences. The cross-run TRT percentage is descriptive only. Amdahl calculations use the sum of mean instrumented phase intervals, not the receiver's paced wall time; they are hypothetical bounds, not measured speedups.

In [3]:
base = medians[("portrait-allocator-ab.json", "baseline")]
optimized = medians[("portrait-allocator-ab.json", "real")]
torch_ten = medians[("portrait-ten-jobs.json", "real")]
trt_ten = medians[("portrait-trt-ten-jobs.json", "real")]
print(f"Matched Torch profile ratio-of-medians gain: {(optimized/base-1)*100:.4f}%")
print(f"UNMATCHED descriptive TRT/Torch ten-job difference: {(trt_ten/torch_ten-1)*100:.4f}%")
phases = phase_means[("portrait-trt-ten-jobs.json", "real")]
groups = {
    "DiT four steps": sum(v for k,v in phases.items() if k.startswith("dit_step_")),
    "decode": phases["decode_0"],
    "motion encode": phases["motion_encode_0"],
    "color": phases["color_0"],
    "audio": phases["audio"],
    "reference-mode allocator-release interval (not weight offload)": phases["dit_offload"],
}
total = sum(phases.values())
for k,v in groups.items():
    print(f"{k}: {v:.3f} ms/chunk, {100*v/total:.2f}% of phase sum")
print(f"Sum of mean instrumented phases: {total:.3f} ms/chunk")
for name in ("DiT four steps", "decode", "motion encode"):
    print(f"HYPOTHETICAL {name} 2x stage speed: {total/(total-groups[name]/2):.4f}x phase-time speedup")
for name in ("audio", "color"):
    print(f"HYPOTHETICAL remove all {name}: {total/(total-groups[name]):.4f}x upper bound")
staged = phase_means[("portrait-9x16-staged.json", "real")]
print(f"Historical staged offload+reload: {staged['dit_offload']+staged['dit_reload']:.3f} ms/chunk")
for demand in (50, 250, 300, 312.5):
    print(f"{demand:g} useful FPS demand / native measured median = {demand/trt_ten:.4f}x")
print(f"+25% throughput = {1.25*trt_ten:.6f} FPS; ten-job generation wall = {2500/(1.25*trt_ten):.6f}s")


Matched Torch profile ratio-of-medians gain: 5.5307%
UNMATCHED descriptive TRT/Torch ten-job difference: 3.5133%
DiT four steps: 441.026 ms/chunk, 55.52% of phase sum
decode: 213.082 ms/chunk, 26.83% of phase sum
motion encode: 73.114 ms/chunk, 9.20% of phase sum
color: 23.785 ms/chunk, 2.99% of phase sum
audio: 14.250 ms/chunk, 1.79% of phase sum
reference-mode allocator-release interval (not weight offload): 17.533 ms/chunk, 2.21% of phase sum
Sum of mean instrumented phases: 794.337 ms/chunk
HYPOTHETICAL DiT four steps 2x stage speed: 1.3843x phase-time speedup
HYPOTHETICAL decode 2x stage speed: 1.1549x phase-time speedup
HYPOTHETICAL motion encode 2x stage speed: 1.0482x phase-time speedup
HYPOTHETICAL remove all audio: 1.0183x upper bound
HYPOTHETICAL remove all color: 1.0309x upper bound
Historical staged offload+reload: 923.076 ms/chunk
50 useful FPS demand / native measured median = 1.7501x
250 useful FPS demand / native measured median = 8.7506x
300 useful FPS demand / native

## Call evidence: transport is not smoothness

Use the same server snapshot's `video_sent` for its held-frame fraction. The soak deliberately used a two-second gap between completed turns; this fraction is not a measure of entirely avoidable scheduler stalls. Generated frames include neural idle and tail padding. Recent first-media summaries are not all-turn percentiles.

In [4]:
soak = reports["calls-generated-idle-soak30m.json"]
s = soak["peers"][0]["server"]
assert soak["all_transport_pass"] and soak["cleanup_pass"]
assert soak["completed_turns"] == 209
assert len(soak["resource_samples"]) == 60
assert s["video_sent"] == 45470 and s["held_frames"] == 10535
assert s["generated_frames"] == 34944 and s["generated_idle_frames"] == 14856
assert soak["total_underrun_frames"] == 23
assert soak["config"]["gap"] == 2.0
assert len(s["turns"]) == 64
print("Soak duration seconds:", round(soak["wall_s"], 6))
print("Soak server-snapshot held fraction:", round(s["held_frames"]/s["video_sent"], 8))
print("Soak receiver frames:", soak["peers"][0]["video_frames"])
print("Soak deliberate between-turn gap seconds:", soak["config"]["gap"])
print("Soak recent turn records / total turn IDs:", len(s["turns"]), s["total_turns"])
ten = reports["calls-h264-trt-c10.json"]
assert ten["total_underrun_frames"] == 9153
print("Ten active transport/cleanup/underruns:",
      ten["all_transport_pass"], ten["cleanup_pass"], ten["total_underrun_frames"])
final = reports["calls-final-staged-c1.json"]
fs = final["peers"][0]["server"]
assert final["total_underrun_frames"] == 251
assert fs["audio_hold_samples"] == 481920
print("Final staged C1 video underruns:", final["total_underrun_frames"])
print("Final staged C1 speech-time audio hold seconds:", fs["audio_hold_samples"]/48000)
print("PASS: historical counters reconciled; no strict real-time claim")


Soak duration seconds: 1819.820635
Soak server-snapshot held fraction: 0.23169122
Soak receiver frames: 45493
Soak deliberate between-turn gap seconds: 2.0
Soak recent turn records / total turn IDs: 64 210
Ten active transport/cleanup/underruns: True True 9153
Final staged C1 video underruns: 251
Final staged C1 speech-time audio hold seconds: 10.04
PASS: historical counters reconciled; no strict real-time claim


## Source coverage and simple contract checks

The inventory selection is all tracked Python, shell, HTML, YAML and requirements files. AST parsing checks syntax/coverage without importing GPU modules. It does not establish runtime correctness. Geometry/queue calculations use the audited contract. Source strings below protect the audit's two concrete helper findings against silently stale documentation; they are not replacements for future regression tests.

In [5]:
paths = subprocess.check_output(
    ["git", "ls-files", "*.py", "*.sh", "*.html", "*.yaml", "*requirements*.txt"],
    cwd=ROOT, text=True).splitlines()
assert len(paths) == 74, len(paths)
lines = sum(len((ROOT/p).read_bytes().splitlines()) for p in paths)
assert lines == 14600, lines
for p in paths:
    if p.endswith(".py"):
        ast.parse((ROOT/p).read_text(), filename=p)
print("Tracked source/config files and physical lines:", len(paths), lines)
recorder = (ROOT/"soulx_rtc/benchmark_rtc.py").read_text()
assert 'vs.width=vs.height=health["size"]' in recorder.replace(" ", "")
vae_source = (ROOT/"flash_head/ltx_video/models/autoencoders/vae.py").read_text()
assert "2 ** (num_blocks - 1)" in vae_source
assert "self.encoder.patch_size_t" in vae_source
installed_block_groups, actual_spatial_stride = 10, 32  # inspected installed Lite config
wrong_latent_tile = int(512 / (2 ** (installed_block_groups - 1)))
right_latent_tile = 512 // actual_spatial_stride
assert wrong_latent_tile == 1 and int(wrong_latent_tile * .75) == 0
print("Tiling helper latent tile / stride-derived tile:", wrong_latent_tile, right_latent_tile)
for w,h in [(512,512),(480,832),(576,1024)]:
    chunk_mib = 24*w*h*3/2**20
    print(f"{w}x{h}: tokens={5*(w//32)*(h//32)}, chunk={chunk_mib:.6f} MiB, "
          f"3 slots/peer={3*chunk_mib:.6f} MiB")
print("Inactive embedding BF16 MiB:", (8653824+1776384)*2/2**20)
print("PASS: inventory syntax and source-derived arithmetic checks")


Tracked source/config files and physical lines: 74 14600


Tiling helper latent tile / stride-derived tile: 1 16
512x512: tokens=1280, chunk=18.000000 MiB, 3 slots/peer=54.000000 MiB
480x832: tokens=1950, chunk=27.421875 MiB, 3 slots/peer=82.265625 MiB
576x1024: tokens=2880, chunk=40.500000 MiB, 3 slots/peer=121.500000 MiB
Inactive embedding BF16 MiB: 19.89404296875
PASS: inventory syntax and source-derived arithmetic checks


## Takeaways and limitations

1. The preserved matched Torch A/B supports a modest profile gain; the separate TRT run does not prove a matched 25% improvement.
2. DiT, decode and motion encoding dominate native-portrait GPU work. Scheduling can remove avoidable waits but cannot supply an unmeasured 8.75× compute gain.
3. Ten-peer transport and a long-running paced connection do not establish ten smooth neural speakers. Holds, underruns and audio insertions remain explicit failures of strict smoothness.
4. Fix recorder geometry and real boundary provenance before making new quality/FPS claims. Repair the VAE tiling contract before enabling it.
5. This notebook validates arithmetic and selected source contracts only. No fresh GPU benchmark, visual evaluation, WAN test or migration certification is implied.

Next action: implement P0 measurement fixes, then the queued-turn/scheduler and profile-guided GPU experiments in [the plan](../NEXT_OPTIMIZATION_PLAN.md).